In [70]:
import pandas as pd
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
import joblib

# NBA Game Prediction Model - KNN

This notebook implements a K-Nearest Neighbors model to predict NBA game outcomes based on team statistics.

In [71]:
# Read interim data
train_data = pd.read_csv('C:/Users/poke5/Desktop/Projects/NBA_Prediction/data/processed/train.csv')
test_data = pd.read_csv('C:/Users/poke5/Desktop/Projects/NBA_Prediction/data/processed/test.csv')
validation_data = pd.read_csv('C:/Users/poke5/Desktop/Projects/NBA_Prediction/data/processed/validation.csv')

In [72]:
def make_matchup_df(df):
    """Create simple feature matrix from games data"""
    features = []
    labels = []
    
    for gid, game in df.groupby('GAME_ID'):
        if len(game) != 2:
            continue
            
        team1 = game.iloc[0]
        team2 = game.iloc[1]
        
        # Get just the rolling features
        feature_cols = [col for col in df.columns if col.startswith('r50_')]
        
        # Calculate differences
        feature_vector = []
        for col in feature_cols:
            diff = float(team1[col]) - float(team2[col])
            feature_vector.append(diff)
            
        # Add home advantage
        feature_vector.append(1 if 'vs.' in team1['MATCHUP'] else 0)
        
        # Add to lists
        features.append(feature_vector)
        labels.append(1 if team1['WL'] == 'W' else 0)
            
    return np.array(features), np.array(labels)

In [73]:
# Create training dataset
X, y = make_matchup_df(train_data)

# Print dataset info
print(f"Dataset shape: {X.shape}")
print(f"Number of games: {len(X)}")
print(f"Number of features: {X.shape[1]}")
print(f"Class balance (wins/losses):", dict(zip(*np.unique(y, return_counts=True))))

Dataset shape: (11650, 9)
Number of games: 11650
Number of features: 9
Class balance (wins/losses): {np.int64(0): np.int64(5821), np.int64(1): np.int64(5829)}


In [74]:
# Train/test split and model training
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale features for KNN (important for distance-based algorithms)
scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(X_tr)
X_val_scaled = scaler.transform(X_val)

In [75]:
# Test different k values to find optimal performance
k_values = range(3, 60, 2)  # Test odd values from 3 to 19
accuracies = []

print('Testing different k values:')
print('-' * 30)
for k in k_values:
    knn_temp = KNeighborsClassifier(n_neighbors=k, weights='distance')
    knn_temp.fit(X_tr_scaled, y_tr)
    y_pred_temp = knn_temp.predict(X_val_scaled)
    acc = accuracy_score(y_val, y_pred_temp)
    accuracies.append(acc)
    print(f'k={k}: Accuracy = {acc:.3f}')

# Find best k
best_k = k_values[np.argmax(accuracies)]
best_acc = max(accuracies)
print(f'\nBest k: {best_k} with accuracy: {best_acc:.3f}')

Testing different k values:
------------------------------
k=3: Accuracy = 0.583
k=5: Accuracy = 0.584
k=3: Accuracy = 0.583
k=5: Accuracy = 0.584
k=7: Accuracy = 0.598
k=7: Accuracy = 0.598
k=9: Accuracy = 0.603
k=11: Accuracy = 0.617
k=9: Accuracy = 0.603
k=11: Accuracy = 0.617
k=13: Accuracy = 0.619
k=15: Accuracy = 0.623
k=13: Accuracy = 0.619
k=15: Accuracy = 0.623
k=17: Accuracy = 0.617
k=17: Accuracy = 0.617
k=19: Accuracy = 0.620
k=19: Accuracy = 0.620
k=21: Accuracy = 0.625
k=21: Accuracy = 0.625
k=23: Accuracy = 0.629
k=23: Accuracy = 0.629
k=25: Accuracy = 0.632
k=25: Accuracy = 0.632
k=27: Accuracy = 0.633
k=27: Accuracy = 0.633
k=29: Accuracy = 0.634
k=29: Accuracy = 0.634
k=31: Accuracy = 0.639
k=31: Accuracy = 0.639
k=33: Accuracy = 0.642
k=33: Accuracy = 0.642
k=35: Accuracy = 0.644
k=35: Accuracy = 0.644
k=37: Accuracy = 0.641
k=37: Accuracy = 0.641
k=39: Accuracy = 0.637
k=39: Accuracy = 0.637
k=41: Accuracy = 0.642
k=41: Accuracy = 0.642
k=43: Accuracy = 0.639
k=43: 

In [76]:
# KNN with optimized parameters for basketball prediction
clf = KNeighborsClassifier(
    n_neighbors = best_k,          
    weights='distance',      
    algorithm='auto',        
    metric='euclidean'       
)

clf.fit(X_tr_scaled, y_tr)
y_pred = clf.predict(X_val_scaled)

# Print evaluation metrics
print('\nModel Evaluation:')
print(f'Training set size: {len(X_tr)} samples')
print(f'Validation set size: {len(X_val)} samples')
print(f'Accuracy on validation set: {accuracy_score(y_val, y_pred):.3f}')
print('\nClassification Report:')
print(classification_report(y_val, y_pred))


Model Evaluation:
Training set size: 9320 samples
Validation set size: 2330 samples
Accuracy on validation set: 0.644

Classification Report:
              precision    recall  f1-score   support

           0       0.65      0.63      0.64      1164
           1       0.64      0.65      0.65      1166

    accuracy                           0.64      2330
   macro avg       0.64      0.64      0.64      2330
weighted avg       0.64      0.64      0.64      2330



In [77]:
def predict_matchup(team1_abbr, team2_abbr, is_team1_home=True):
    """Simple prediction for a matchup between two teams using KNN"""
    # Get most recent stats for both teams
    team1_data = train_data[train_data['TEAM_ABBREVIATION'] == team1_abbr].iloc[-1]
    team2_data = train_data[train_data['TEAM_ABBREVIATION'] == team2_abbr].iloc[-1]
    
    # Just use the basic rolling stats and home advantage
    features_to_use = [col for col in train_data.columns if col.startswith('r50_')]
    
    # Build feature vector (differences between teams)
    feature_vector = []
    for col in features_to_use:
        diff = float(team1_data[col]) - float(team2_data[col])
        feature_vector.append(diff)
    
    # Add home court advantage
    feature_vector.append(1 if is_team1_home else 0)
    
    # Scale the features (important for KNN)
    X_pred = np.array([feature_vector])
    X_pred_scaled = scaler.transform(X_pred)
    
    # Make prediction
    prob = clf.predict_proba(X_pred_scaled)[0][1]
    pred = clf.predict(X_pred_scaled)[0]
    winner = team1_abbr if pred == 1 else team2_abbr
    

In [78]:
# Calculate team averages using the latest stats for each team
features_to_use = [col for col in train_data.columns if col.startswith('r50_')]
team_avgs = {}

for team in train_data['TEAM_ABBREVIATION'].unique():
    team_recent = train_data[train_data['TEAM_ABBREVIATION'] == team].iloc[-1]
    team_avgs[team] = team_recent[features_to_use]

team_avgs = pd.DataFrame(team_avgs).T 
print("KNN Team averages shape:", team_avgs.shape)
print("\nFirst few rows:")
print(team_avgs.head())

KNN Team averages shape: (32, 8)

First few rows:
    r50_MissedFG r50_MissedFT  r50_TSA r50_TS_Pct r50_EffIndex r50_FG_Eff  \
POR        50.16          4.2  101.316   56.08406       88.436   0.450623   
HOU        49.44         5.52  101.812  57.990289       91.436   0.449774   
BOS        47.24          4.3  98.5728  57.577617       96.246   0.462101   
MIA        45.14         5.26  95.5744  58.470106        93.18   0.462541   
UTA        45.08          5.3  95.7472   58.72241       88.912   0.469277   

    r50_RebRatio    r50_TS  
POR     0.223099  0.560841  
HOU     0.221566  0.579903  
BOS     0.219733  0.575776  
MIA     0.185901  0.584701  
UTA     0.195822  0.587224  


In [79]:
# Save KNN model, scaler, and team averages for PredictionSimulator
joblib.dump(clf, 'knn_model.pkl')
joblib.dump(scaler, 'knn_scaler.pkl') 
team_avgs.to_csv('C:/Users/poke5/Desktop/Projects/NBA_Prediction/data/processed/team_averages_knn.csv')